### Dependencias

In [1]:
import cv2
import os
import numpy as np
import mediapipe as mp
import tensorflow as tf

from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.layers import Attention
from tensorflow.keras.layers import GlobalAveragePooling1D

### Configuración

In [ ]:
# DATA
DATASET_PATH = "dataset"
MODEL_PATH = "models"

# SIGNS TO COLLECT
SIGNS = ["J"]

# SEQUENCE
SEQUENCE_LENGTH = 50
NUM_SEQUENCES = 50

# FEATURES
LANDMARKS = 21
COORDS = 3
BASE_FEATURES = LANDMARKS * COORDS
TOTAL_FEATURES = BASE_FEATURES * 2  # coords + velocity

### Inicio de MediaPipe

In [4]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

### Normalización

In [5]:
def normalize_landmarks(landmarks):

    coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])

    wrist = coords[0]

    coords = coords - wrist

    scale = np.linalg.norm(coords[9])

    if scale != 0:
        coords = coords / scale

    return coords.flatten()

### Captura y vector final

In [6]:
# Movimiento entre Frames
def compute_velocity(curr, prev):

    if prev is None:
        return np.zeros_like(curr)

    return curr - prev

In [7]:
# Vector final
def extract_features(hand_landmarks, prev_features):

    coords = normalize_landmarks(hand_landmarks.landmark)

    velocity = compute_velocity(coords, prev_features)

    features = np.concatenate([coords, velocity])

    return features, coords

### Recolección de datos

In [8]:
def collect_dataset(sign):

    os.makedirs(f"{DATASET_PATH}/{sign}", exist_ok=True)

    cap = cv2.VideoCapture(0)

    sequence = []
    prev_features = None
    sequence_id = 0

    while sequence_id < NUM_SEQUENCES:

        ret, frame = cap.read()

        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(image)

        if results.multi_hand_landmarks:

            hand_landmarks = results.multi_hand_landmarks[0]

            features, coords = extract_features(hand_landmarks, prev_features)

            prev_features = coords

            sequence.append(features)

            mp_drawing.draw_landmarks(
                frame,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS
            )

            if len(sequence) == SEQUENCE_LENGTH:

                np.save(
                    f"{DATASET_PATH}/{sign}/{sequence_id}.npy",
                    np.array(sequence)
                )

                print("Saved sequence", sequence_id)

                sequence = []
                sequence_id += 1

        cv2.imshow("Capture", frame)

        if cv2.waitKey(10) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

In [9]:
for sign in SIGNS:
    collect_dataset(sign)

Saved sequence 0
Saved sequence 1
Saved sequence 2
Saved sequence 3
Saved sequence 4
Saved sequence 5
Saved sequence 6
Saved sequence 7
Saved sequence 8
Saved sequence 9
Saved sequence 10
Saved sequence 11
Saved sequence 12
Saved sequence 13
Saved sequence 14
Saved sequence 15
Saved sequence 16
Saved sequence 17
Saved sequence 18
Saved sequence 19
Saved sequence 20
Saved sequence 21
Saved sequence 22
Saved sequence 23
Saved sequence 24
Saved sequence 25
Saved sequence 26
Saved sequence 27
Saved sequence 28
Saved sequence 29
Saved sequence 30
Saved sequence 31
Saved sequence 32
Saved sequence 33
Saved sequence 34
Saved sequence 35
Saved sequence 36
Saved sequence 37
Saved sequence 38
Saved sequence 39
Saved sequence 40
Saved sequence 41
Saved sequence 42
Saved sequence 43
Saved sequence 44
Saved sequence 45
Saved sequence 46
Saved sequence 47
Saved sequence 48
Saved sequence 49
Saved sequence 50
Saved sequence 51
Saved sequence 52
Saved sequence 53
Saved sequence 54
Saved sequence 55
Sa

### Crear Dataset

In [10]:
def build_dataset():

    X = []
    y = []

    labels = os.listdir(DATASET_PATH)

    label_map = {label: i for i, label in enumerate(labels)}

    for label in labels:

        for file in os.listdir(f"{DATASET_PATH}/{label}"):

            seq = np.load(f"{DATASET_PATH}/{label}/{file}")

            X.append(seq)
            y.append(label_map[label])

    return np.array(X), np.array(y), label_map
# Cargar Dataset
X, y, label_map = build_dataset()

print("Dataset shape:", X.shape)

Dataset shape: (100, 30, 126)


### Crear Modelo .Keras

In [11]:
def create_model(num_classes):

    inputs = tf.keras.Input(shape=(SEQUENCE_LENGTH, TOTAL_FEATURES))

    x = LSTM(128, return_sequences=True)(inputs)
    x = Dropout(0.3)(x)

    x = LSTM(128, return_sequences=True)(x)

    attention = Attention()([x, x])

    x = GlobalAveragePooling1D()(attention)

    x = Dense(64, activation="relu")(x)

    outputs = Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

### Training

In [12]:
model = create_model(len(label_map))

model.fit(
    X,
    y,
    epochs=50,
    batch_size=32,
    validation_split=0.2
)
#Guardar modelo
os.makedirs(MODEL_PATH, exist_ok=True)

model.save(f"{MODEL_PATH}/sign_model.keras")

Epoch 1/50
3/3 [==============================] - 6s 585ms/step - loss: 0.0000e+00 - accuracy: 0.0000e+00 - val_loss: 0.0000e+00 - val_accuracy: 0.0000e+00
Epoch 2/50
3/3 [==============================] - 0s 110ms/step - loss: 0.0000e+00 - accuracy: 0.0000e+00 - val_loss: 0.0000e+00 - val_accuracy: 0.0000e+00
Epoch 3/50
3/3 [==============================] - 0s 99ms/step - loss: 0.0000e+00 - accuracy: 0.0000e+00 - val_loss: 0.0000e+00 - val_accuracy: 0.0000e+00
Epoch 4/50
3/3 [==============================] - 0s 99ms/step - loss: 0.0000e+00 - accuracy: 0.0000e+00 - val_loss: 0.0000e+00 - val_accuracy: 0.0000e+00
Epoch 5/50
3/3 [==============================] - 0s 92ms/step - loss: 0.0000e+00 - accuracy: 0.0000e+00 - val_loss: 0.0000e+00 - val_accuracy: 0.0000e+00
Epoch 6/50
3/3 [==============================] - 0s 89ms/step - loss: 0.0000e+00 - accuracy: 0.0000e+00 - val_loss: 0.0000e+00 - val_accuracy: 0.0000e+00
Epoch 7/50
3/3 [==============================] - 0s 97ms/step - los

### Exportación

In [14]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

with open(f"{MODEL_PATH}/sign_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved")

INFO:tensorflow:Assets written to: C:\Users\ledys\AppData\Local\Temp\tmpiye5kz6e\assets


INFO:tensorflow:Assets written to: C:\Users\ledys\AppData\Local\Temp\tmpiye5kz6e\assets


TFLite model saved


### Test Local

In [15]:
sequence = []
prev_features = None

cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(image)

    if results.multi_hand_landmarks:

        hand_landmarks = results.multi_hand_landmarks[0]

        features, coords = extract_features(hand_landmarks, prev_features)

        prev_features = coords

        sequence.append(features)

        sequence = sequence[-SEQUENCE_LENGTH:]

        if len(sequence) == SEQUENCE_LENGTH:

            input_data = np.expand_dims(sequence, axis=0)

            prediction = model.predict(input_data)

            class_id = np.argmax(prediction)

            confidence = np.max(prediction)

            if confidence > 0.8:

                sign = list(label_map.keys())[class_id]

                cv2.putText(
                    frame,
                    f"{sign} {confidence:.2f}",
                    (10,40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0,255,0),
                    2
                )

        mp_drawing.draw_landmarks(
            frame,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS
        )

    cv2.imshow("Inference", frame)

    if cv2.waitKey(10) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

1/1 [==============================] - 0s 33ms/step
